# Kaggle Phase 1 — Data, Synthetic Bursts, Adaptive Windows

First of 3 notebooks (`kagglephase1` → `kagglephase2` → `kagglephase3`).

**What this notebook produces** (all small files, ~60MB total, in `/kaggle/working`):
- `features_injected.npy` — complex_case1, normalized + feature-engineered, with synthetic bursts
- `features_clean.npy` — same rows without injection (for the untouched-data comparison)
- `windows_train/val/test.npy` — rolling-origin window tables `[anchor_end, adaptive_length]`
- `spike_mask.npy`, `spike_events.json` — exactly which rows were modified
- `segments.json`, `feature_cols.json`, `normalization_stats.json`, `manifest.json`
- `kagglephase1_output.zip` — everything above in one archive

**Design notes**
- Single-case approach retained: complex_case1 only.
- Adaptive sliding window 500–1000 (proposal component): with a 1000-step max lookback,
  a hard purge-gap split would leave val/test (15% slices ≈ 1,243 rows/container) with
  zero valid windows — so this pipeline uses standard rolling-origin evaluation instead:
  a window's LOOKBACK may reach back into earlier rows as context, but its TARGET rows
  decide which split it belongs to, with a 10-row embargo so no target is ever shared
  between splits. Context-from-the-past is standard forecasting practice, not leakage.
- Low memory: one continuous `(n_rows, 27)` float32 feature array per variant (~24MB);
  windows are slices built on the fly downstream — no multi-GB X files.

**After running:** download `kagglephase1_output.zip` from the Output tab and upload it
(unzipped) as a Kaggle Dataset named e.g. `kagglephase1-output`, then attach that to
`kagglephase2` / `kagglephase3` via Add Input.


## Step 1: Platform + Input Check

In [4]:
import sys, os, gc, json
import numpy as np
import pandas as pd
from pathlib import Path

IN_COLAB  = os.path.exists('/var/colab/hostname')
IN_KAGGLE = os.path.exists('/kaggle')
print("Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local")

if IN_KAGGLE:
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists() or not any(kaggle_input.iterdir()):
        raise FileNotFoundError(
            "/kaggle/input is empty -- attach the raw-data dataset (the one containing "
            "raw/complex/case1/container/kpi_*.csv) via 'Add Input' in the right sidebar, "
            "then re-run this cell."
        )
    _cands = sorted(kaggle_input.glob('**/kpi_container_cpu_usage_seconds_total.csv'),
                    key=lambda p: len(p.parts))
    if not _cands:
        raise FileNotFoundError(f"No kpi_container_cpu_usage_seconds_total.csv found under {kaggle_input} "
                                f"-- is the correct raw-data dataset attached?")
    raw_path = _cands[0].parent
    while raw_path.name not in ('complex', 'single') and raw_path != raw_path.parent:
        raw_path = raw_path.parent
    if raw_path.name in ('complex', 'single'):
        raw_path = raw_path.parent
else:
    raw_path = Path('./raw')

case1_dir = raw_path / 'complex' / 'case1' / 'container'
if not case1_dir.exists():
    raise FileNotFoundError(f"{case1_dir} not found -- expected <raw_root>/complex/case1/container/")
print(f"Raw complex_case1 dir: {case1_dir} ({len(list(case1_dir.glob('kpi_*.csv')))} metric files)")

OUT_DIR = Path('/kaggle/working/phase1_output') if IN_KAGGLE else Path('./phase1_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)


Kaggle
Raw complex_case1 dir: /kaggle/input/datasets/thanakaran/raw-data/raw/complex/case1/container (17 metric files)


## Step 2: Load complex_case1 and Pivot to Wide Format

In [5]:
RAW_METRICS = [
    'container_cpu_usage_seconds_total',
    'container_cpu_system_seconds_total',
    'container_cpu_user_seconds_total',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
    'container_memory_cache',
]
TARGET_COLUMNS = [
    'container_cpu_usage_seconds_total', 'container_memory_usage_bytes',
    'container_memory_working_set_bytes', 'container_memory_rss',
]
TARGET_NAMES = ['cpu_usage', 'mem_usage', 'mem_working_set', 'mem_rss']

dfs = []
for f in sorted(case1_dir.glob('kpi_*.csv')):
    d = pd.read_csv(f)
    if 'kpi_name' in d.columns:
        d = d[d['kpi_name'].isin(RAW_METRICS)]
    if len(d):
        dfs.append(d)
case_df = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()

case_df = case_df.pivot_table(index=['timestamp', 'cmdb_id'], columns='kpi_name',
                              values='value', aggfunc='first').reset_index()
case_df.columns.name = None
missing = [c for c in RAW_METRICS if c not in case_df.columns]
if missing:
    raise ValueError(f"Missing metric columns after pivot: {missing}")
for c in RAW_METRICS:
    case_df[c] = pd.to_numeric(case_df[c], errors='coerce').astype(np.float64)
case_df.sort_values(['cmdb_id', 'timestamp'], inplace=True)
case_df.reset_index(drop=True, inplace=True)
print(f"complex_case1: {len(case_df):,} rows, {case_df['cmdb_id'].nunique()} containers, "
      f"{case_df[RAW_METRICS].isna().sum().sum()} NaNs in metric columns")


complex_case1: 223,830 rows, 27 containers, 0 NaNs in metric columns


## Step 3: Artifact Check on the Natural 'Spikes' (Phase 0 of the plan)

The all-4-case scan (v9) found max/p95 step ratios of 200–1600x, with identical extreme
values recurring across unrelated cases — a signature of collection artifacts (counter
resets / monitoring gaps), not real workload bursts. This step inspects the largest step
per target directly so the synthetic burst design below is grounded in verified behavior,
not glitches.


In [6]:
for col, name in zip(TARGET_COLUMNS, TARGET_NAMES):
    steps, ends, cids = [], [], []
    for cid, g in case_df.groupby('cmdb_id'):
        v = g[col].values
        if len(v) > 1:
            s = np.abs(np.diff(v))
            steps.append(s); ends.append(g.index.values[1:]); cids.append(np.repeat(cid, len(s)))
    steps = np.concatenate(steps); ends = np.concatenate(ends); cids = np.concatenate(cids)
    k = int(np.argmax(steps))
    i = int(ends[k])
    ctx = case_df.loc[max(0, i-3):i+2, ['timestamp', 'cmdb_id', col]]
    p95 = np.percentile(steps, 95)
    print(f"--- {name}: max step={steps[k]:,.2f} (p95={p95:,.2f}, ratio={steps[k]/max(p95,1e-9):.0f}x) "
          f"container={cids[k]} ---")
    print(ctx.to_string(index=False))
    frac_drop = (np.diff(case_df[case_df.cmdb_id == cids[k]][col].values) < -p95*5).mean()
    print(f"    large NEGATIVE steps in that container (counter-reset signature): {frac_drop*100:.3f}% of steps")
    print()
print("Verdict guidance: isolated one-off jumps with matching magnitudes across containers/cases,")
print("or paired drop-then-recover patterns, are collection artifacts -- the synthetic burst design")
print("below therefore anchors magnitudes to the p95/p99 of NORMAL steps, never to these maxima.")


--- cpu_usage: max step=1,916.11 (p95=1.16, ratio=1648x) container=observe.frontend-0 ---
 timestamp            cmdb_id  container_cpu_usage_seconds_total
1719212790 observe.frontend-0                           1921.067
1719212805 observe.frontend-0                           1921.067
1719212820 observe.frontend-0                           1921.067
1719213135 observe.frontend-0                              4.956
1719213150 observe.frontend-0                              4.956
1719213165 observe.frontend-0                              4.956
    large NEGATIVE steps in that container (counter-reset signature): 0.012% of steps

--- mem_usage: max step=107,339,776.00 (p95=421,888.00, ratio=254x) container=observe.frontend-1 ---
 timestamp            cmdb_id  container_memory_usage_bytes
1719271950 observe.frontend-1                   184430592.0
1719271965 observe.frontend-1                   184430592.0
1719271980 observe.frontend-1                   184430592.0
1719271995 observe.frontend

## Step 4: Split Boundaries (rolling-origin, target-row based, 10-row embargo)

In [7]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
MIN_LOOKBACK, MAX_LOOKBACK = 500, 1000
MAX_HORIZON = 10

containers = case_df['cmdb_id'].unique().tolist()
segments = {}
_row = 0
for cid in containers:
    n = int((case_df['cmdb_id'] == cid).sum())
    segments[cid] = {'start': _row, 'end': _row + n,
                     'i70': _row + int(n * TRAIN_FRAC),
                     'i85': _row + int(n * (TRAIN_FRAC + VAL_FRAC))}
    _row += n
n_rows_total = _row
assert n_rows_total == len(case_df)

for cid in containers[:3]:
    s = segments[cid]
    print(f"{cid}: rows [{s['start']}, {s['end']})  train<{s['i70']}  val<{s['i85']}")
print(f"... ({len(containers)} containers, {n_rows_total:,} rows total)")


observe.cartservice-0: rows [0, 8302)  train<5811  val<7056
observe.cartservice-1: rows [8302, 16583)  train<14098  val<15340
observe.cartservice-2: rows [16583, 24867)  train<22381  val<23624
... (27 containers, 223,830 rows total)


## Step 5: Synthetic Burst Injection

Grounded in the artifact check + v9 statistics (cpu p95 step ≈ 1.2 on values ~400–1450;
memory p95 step ≈ 420KB on levels ~0.6–1.05e8):

- **cpu_usage** is a cumulative counter → a burst is a temporarily steeper SLOPE (extra
  rate ramps up 20 steps, holds 10, ramps down 20; the counter stays permanently higher
  by the burst's integral — monotonicity preserved, as physics requires). Peak extra rate
  is drawn from 4–8× the container's own p95 natural step.
- **memory metrics** are levels → a burst is a bump-and-return (same 20/10/20 ramp/hold/
  decay), peak 15–35% of the container's median level, applied to all 3 memory targets
  with correlated (0.8–1.0×) magnitudes.
- **Distinct instances per split**: separate seeded RNGs (train=101, val=202, test=303)
  choose positions/magnitudes inside each split's own row region, ≥300 steps apart.
  Train sees 8 events/container, val 3, test 4 — same class of pattern, never the same event.
- Every modified row is recorded in `spike_mask` + `spike_events.json`.


In [8]:
RAMP_UP, HOLD, RAMP_DOWN = 20, 10, 20
EVENT_LEN = RAMP_UP + HOLD + RAMP_DOWN
EVENTS_PER_SPLIT = {'train': 8, 'val': 3, 'test': 4}
SPLIT_SEEDS     = {'train': 101, 'val': 202, 'test': 303}
MIN_SEPARATION  = 300

def _event_profile():
    up   = np.linspace(0, 1, RAMP_UP, endpoint=False)
    hold = np.ones(HOLD)
    down = np.linspace(1, 0, RAMP_DOWN)
    return np.concatenate([up, hold, down])

def gen_event_positions(rng, lo, hi, k, min_sep=MIN_SEPARATION):
    positions = []
    for _ in range(200):
        if len(positions) >= k:
            break
        p = int(rng.integers(lo, max(lo + 1, hi - EVENT_LEN)))
        if all(abs(p - q) >= min_sep for q in positions):
            positions.append(p)
    return sorted(positions)

def inject_bursts(values_by_col, seg, split_name, rng, cpu_col, mem_cols):
    lo = {'train': seg['start'] + 50, 'val': seg['i70'], 'test': seg['i85']}[split_name]
    hi = {'train': seg['i70'] - MAX_HORIZON, 'val': seg['i85'] - MAX_HORIZON,
          'test': seg['end'] - MAX_HORIZON}[split_name]
    prof = _event_profile()
    events = []
    cpu = values_by_col[cpu_col]
    nat_step = np.abs(np.diff(cpu[seg['start']:seg['i70']]))
    cpu_p95 = max(np.percentile(nat_step, 95), 1e-6)
    for p in gen_event_positions(rng, lo, hi, EVENTS_PER_SPLIT[split_name]):
        e = {'start': int(p), 'end': int(p + EVENT_LEN), 'split': split_name, 'targets': {}}
        rate_peak = float(rng.uniform(4.0, 8.0) * cpu_p95)
        extra = np.cumsum(prof * rate_peak)
        # permanent counter shift is bounded to THIS container's segment --
        # a global-tail shift would corrupt every later container's rows
        cpu[p + EVENT_LEN:seg['end']] += extra[-1]
        cpu[p:p + EVENT_LEN] += extra
        e['targets'][cpu_col] = {'kind': 'rate', 'peak_extra_rate': rate_peak}
        base_frac = float(rng.uniform(0.15, 0.35))
        for mc in mem_cols:
            level = float(np.median(values_by_col[mc][seg['start']:seg['i70']]))
            corr = float(rng.uniform(0.8, 1.0))
            peak = base_frac * corr * level
            values_by_col[mc][p:p + EVENT_LEN] += prof * peak
            e['targets'][mc] = {'kind': 'bump', 'peak': peak}
        events.append(e)
    return events

clean_raw = {c: case_df[c].values.copy() for c in RAW_METRICS}
inj_raw   = {c: case_df[c].values.copy() for c in RAW_METRICS}
spike_mask = np.zeros(n_rows_total, dtype=bool)
all_events = []
cpu_col = 'container_cpu_usage_seconds_total'
mem_cols = ['container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss']

for split_name in ('train', 'val', 'test'):
    rng = np.random.default_rng(SPLIT_SEEDS[split_name])
    for cid in containers:
        evts = inject_bursts(inj_raw, segments[cid], split_name, rng, cpu_col, mem_cols)
        for e in evts:
            e['container'] = cid
            spike_mask[e['start']:e['end']] = True
        all_events.extend(evts)

for cid in containers:
    s = segments[cid]
    d = np.diff(inj_raw[cpu_col][s['start']:s['end']])
    neg_before = (np.diff(clean_raw[cpu_col][s['start']:s['end']]) < 0).sum()
    assert (d < 0).sum() <= neg_before, f"cpu monotonicity worsened by injection for {cid}"

by_split = pd.Series([e['split'] for e in all_events]).value_counts().to_dict()
print(f"Injected {len(all_events)} burst events across {len(containers)} containers: {by_split}")
print(f"Rows inside burst regions: {spike_mask.sum():,} / {n_rows_total:,} ({spike_mask.mean()*100:.2f}%)")


Injected 389 burst events across 27 containers: {'train': 216, 'test': 92, 'val': 81}
Rows inside burst regions: 19,450 / 223,830 (8.69%)


## Step 6: Normalize (train-only stats) + Feature Engineering (both variants)

In [9]:
train_row_mask = np.zeros(n_rows_total, dtype=bool)
for cid in containers:
    s = segments[cid]
    train_row_mask[s['start']:s['i70']] = True

train_stats = {}
for c in RAW_METRICS:
    v = inj_raw[c][train_row_mask]
    m, sd = float(np.mean(v)), float(np.std(v))
    train_stats[c] = {'mean': m, 'std': sd if sd > 0 else 1.0}

def build_features(raw_by_col):
    norm = {c: (raw_by_col[c] - train_stats[c]['mean']) / train_stats[c]['std'] for c in RAW_METRICS}
    cols, names = [], []
    for c in RAW_METRICS:
        cols.append(norm[c].astype(np.float32)); names.append(c)
    for c in TARGET_COLUMNS:
        v = norm[c]
        for lag in (1, 2, 3):
            d = np.zeros_like(v)
            for cid in containers:
                s = segments[cid]
                seg = v[s['start']:s['end']]
                d[s['start'] + lag:s['end']] = seg[lag:] - seg[:-lag]
            cols.append(d.astype(np.float32)); names.append(f'{c}_DIFF_{lag}')
        rm = np.zeros_like(v); rs = np.zeros_like(v)
        for cid in containers:
            s = segments[cid]
            seg = pd.Series(v[s['start']:s['end']])
            rm[s['start']:s['end']] = seg.rolling(3, min_periods=1).mean().values
            rs[s['start']:s['end']] = seg.rolling(3, min_periods=1).std().fillna(0).values
        cols.append(rm.astype(np.float32)); names.append(f'{c}_ROLLING_MEAN_3')
        cols.append(rs.astype(np.float32)); names.append(f'{c}_ROLLING_STD_3')
    return np.stack(cols, axis=1), names

features_injected, feature_cols = build_features(inj_raw)
features_clean, _ = build_features(clean_raw)
target_idx = [feature_cols.index(c) for c in TARGET_COLUMNS]
assert features_injected.shape == (n_rows_total, 27), features_injected.shape
assert not np.isnan(features_injected).any() and not np.isnan(features_clean).any()
print(f"features: {features_injected.shape} float32  ({features_injected.nbytes/1e6:.1f} MB per variant)")
print(f"target columns at indices {dict(zip(TARGET_NAMES, target_idx))}")


features: (223830, 27) float32  (24.2 MB per variant)
target columns at indices {'cpu_usage': 0, 'mem_usage': 3, 'mem_working_set': 4, 'mem_rss': 5}


## Step 7: Adaptive-Length Rolling-Origin Window Tables

Adaptive sliding window (proposal component): per anchor, lookback length L ∈ [500, 1000]
set by recent workload variability — rolling std of normalized cpu over the last 120 steps,
compared to the container's train-period median of that same statistic. Stable ⇒ long
window (more context); volatile ⇒ short window (recent data dominates), following the
ADWIN shrink-on-change convention. Split membership is decided by target rows
(anchor+1 … anchor+10), with train capped 10 rows before the val region (embargo).


In [10]:
VAR_WIN = 120

def adaptive_lengths(cpu_norm_col):
    roll = pd.Series(cpu_norm_col).rolling(VAR_WIN, min_periods=20).std().fillna(0).values
    L = np.full(n_rows_total, MAX_LOOKBACK, dtype=np.int32)
    for cid in containers:
        s = segments[cid]
        ref = np.median(roll[s['start'] + VAR_WIN:s['i70']])
        ref = max(ref, 1e-9)
        r = roll[s['start']:s['end']] / ref
        Lc = np.clip(np.round(MAX_LOOKBACK - 250 * (r - 1.0)), MIN_LOOKBACK, MAX_LOOKBACK)
        L[s['start']:s['end']] = Lc.astype(np.int32)
    return L

def build_windows(lengths):
    out = {'train': [], 'val': [], 'test': []}
    for cid in containers:
        s = segments[cid]
        for anchor in range(s['start'] + MIN_LOOKBACK - 1, s['end'] - MAX_HORIZON):
            L = min(int(lengths[anchor]), anchor + 1 - s['start'])
            if L < MIN_LOOKBACK:
                continue
            end = anchor + 1
            if end + MAX_HORIZON - 1 < s['i70'] - MAX_HORIZON:
                split = 'train'
            elif end >= s['i70'] and end + MAX_HORIZON - 1 < s['i85']:
                split = 'val'
            elif end >= s['i85']:
                split = 'test'
            else:
                continue
            out[split].append((end, L))
    return {k: np.asarray(v, dtype=np.int32) for k, v in out.items()}

lengths_inj = adaptive_lengths(features_injected[:, feature_cols.index(cpu_col)])
windows = build_windows(lengths_inj)
lengths_clean = adaptive_lengths(features_clean[:, feature_cols.index(cpu_col)])
windows_clean_test = build_windows(lengths_clean)['test']

for k, w in windows.items():
    print(f"{k:>6}: {len(w):,} windows | length min/mean/max = "
          f"{w[:,1].min()}/{w[:,1].mean():.0f}/{w[:,1].max()}")
print(f"clean test: {len(windows_clean_test):,} windows")
n_short = (windows['train'][:,1] < MAX_LOOKBACK).sum()
print(f"train windows shortened by variability rule: {n_short:,} ({n_short/len(windows['train'])*100:.1f}%)")


 train: 142,653 windows | length min/mean/max = 500/887/1000
   val: 33,335 windows | length min/mean/max = 500/833/1000
  test: 33,343 windows | length min/mean/max = 500/809/1000
clean test: 33,343 windows
train windows shortened by variability rule: 73,591 (51.6%)


## Step 8: Save Outputs + Package

In [11]:
import shutil, zipfile

np.save(OUT_DIR / 'features_injected.npy', features_injected)
np.save(OUT_DIR / 'features_clean.npy', features_clean)
for k, w in windows.items():
    np.save(OUT_DIR / f'windows_{k}.npy', w)
np.save(OUT_DIR / 'windows_test_clean.npy', windows_clean_test)
np.save(OUT_DIR / 'spike_mask.npy', spike_mask)

with open(OUT_DIR / 'spike_events.json', 'w') as f:
    json.dump(all_events, f, indent=1)
with open(OUT_DIR / 'segments.json', 'w') as f:
    json.dump({str(c): segments[c] for c in containers}, f, indent=1)
with open(OUT_DIR / 'feature_cols.json', 'w') as f:
    json.dump({'feature_cols': feature_cols, 'target_columns': TARGET_COLUMNS,
               'target_names': TARGET_NAMES, 'target_idx': target_idx}, f, indent=1)
with open(OUT_DIR / 'normalization_stats.json', 'w') as f:
    json.dump(train_stats, f, indent=1)
with open(OUT_DIR / 'manifest.json', 'w') as f:
    json.dump({
        'pipeline': 'kagglephase1', 'case': 'complex_case1',
        'design': 'rolling-origin, target-row split, 10-row embargo, adaptive lookback',
        'min_lookback': MIN_LOOKBACK, 'max_lookback': MAX_LOOKBACK, 'max_horizon': MAX_HORIZON,
        'window_format': '[anchor_end_exclusive, lookback_length]; X = features[end-L:end]; '
                         'y(h) = features[end-1+h, target_idx]',
        'burst_design': {'profile': f'ramp{RAMP_UP}/hold{HOLD}/decay{RAMP_DOWN}',
                         'cpu': 'rate-bump (monotonic, permanent level shift)',
                         'memory': 'bump-and-return, 15-35% of median level, correlated x0.8-1.0',
                         'events_per_split': EVENTS_PER_SPLIT, 'seeds': SPLIT_SEEDS},
        'n_rows': int(n_rows_total), 'n_containers': len(containers),
    }, f, indent=1)

zip_path = OUT_DIR.parent / 'kagglephase1_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.iterdir()):
        z.write(p, p.name)

total_mb = sum(p.stat().st_size for p in OUT_DIR.iterdir()) / 1e6
print(f"Wrote {len(list(OUT_DIR.iterdir()))} files ({total_mb:.1f} MB) to {OUT_DIR}")
print(f"Archive: {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)")
print()
print("NEXT: Output tab -> download kagglephase1_output.zip -> unzip -> upload the files")
print("as a Kaggle Dataset named 'kagglephase1-output' -> attach it to kagglephase2/3 via Add Input.")


Wrote 12 files (50.7 MB) to /kaggle/working/phase1_output
Archive: /kaggle/working/kagglephase1_output.zip (13.6 MB)

NEXT: Output tab -> download kagglephase1_output.zip -> unzip -> upload the files
as a Kaggle Dataset named 'kagglephase1-output' -> attach it to kagglephase2/3 via Add Input.
